# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides tools for loading and exploring the FAIRˆ² dataset (mass spectrometry data) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Dataset title: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"DOI: {getattr(metadata, 'identifier', None)}")
print(f"Version: {getattr(metadata, 'version', None)}\n")

## 2. Data Overview
Review available record sets, their `@id`s, corresponding fields, and their `@id`s.

In [ ]:
from pprint import pprint

# List all available record sets and their field @ids
print('RecordSets available in the dataset:')
ds = dataset  # short alias
record_sets = list(ds.record_sets)
for rec in record_sets:
    print(f"  - @id: {rec['@id']}, name: {rec.get('name', '')}")

print('\nListing fields for each RecordSet:')
for rec in record_sets:
    print(f"\nRecordSet: {rec['@id']}")
    fields = rec.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    for field in fields:
        # field is an @id reference; lookup actual field definition
        field_obj = ds._metadata.resolve(field['@id']) if isinstance(field, dict) and '@id' in field else field
        if field_obj and '@id' in field_obj:
            print(f"    - {field_obj['@id']}: {field_obj.get('name', '')} ({field_obj.get('dataType', '')})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Refer to the record set and field `@id`s listed above.

In [ ]:
# List all record set @ids for extraction
record_set_ids = [rec['@id'] for rec in ds.record_sets]
dataframes = {}

# Extract all records in all record sets to dataframes
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for RecordSet {record_set_id}, columns: {df.columns.tolist()}")
    else:
        print(f"No records found for RecordSet {record_set_id}")

# For demonstration, pick the first record set with data for further analysis
main_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        main_record_set_id = k
        break
if main_record_set_id:
    print(f"\nUsing RecordSet {main_record_set_id} for analysis. Columns:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# --- Identify numeric field candidates ---
df = dataframes[main_record_set_id]
numeric_field_id = None

# Attempt to pick a suggestive field name for demonstration
for col in df.columns:
    if 'MW' in col or 'Abundance' in col or 'Coverage' in col:
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback: find a numeric column by dtype
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Using numeric field for filtering: {numeric_field_id}")

threshold = df[numeric_field_id].mean()  # Use mean as an arbitrary threshold for demonstration
filtered_df = df[df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (mean value):")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()

print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a likely categorical field (e.g., Accession, Description, or 'Sample')
group_field = None
for cname in ['Accession', 'Description', 'Sample']:
    if cname in df.columns:
        group_field = cname
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
    print(f"Mean {numeric_field_id} grouped by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt

# Histogram of numeric field
plt.figure(figsize=(8, 5))
df[numeric_field_id].hist(bins=30)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# If we have a group_field with a reasonable number of unique values, make a boxplot
if group_field and df[group_field].nunique() <= 20:
    plt.figure(figsize=(12, 5))
    df.boxplot(column=numeric_field_id, by=group_field, rot=45)
    plt.title(f"{numeric_field_id} by {group_field}")
    plt.suptitle('')
    plt.xlabel(group_field)
    plt.ylabel(numeric_field_id)
    plt.show()

## 6. Conclusion
This notebook demonstrates end-to-end exploration of a mass spectrometry protein dataset using the `mlcroissant` library.

- We loaded metadata and identified record sets and fields by their `@id`s.
- We loaded and previewed one record set, performed filtering, normalization, and grouping, and visualized the main numeric field.
- The Croissant schema ensures reliable field and data reference by `@id`, making future analyses and data integration robust.

You can further refine the above analysis by referencing the actual data dictionary and using more of the fields and record sets revealed in section 2.